# Mereska — TikTok / RedNote Dubbing (Kaggle)

Notebook khusus video pendek yang sudah selesai diedit. Alur: **ingest → analisis visual → storytelling Indonesia → Edge TTS → word timing → subtitle animasi → MP4**.

Default: **seluruh video asli**, rasio `source`, suara **Gadis**, subtitle **karaoke**, dan **canvas tambahan nonaktif**. Tidak ada pemilihan/penyusunan ulang footage summary.

Sebelum menjalankan:
1. Aktifkan **Internet** dan GPU Kaggle untuk Qwen. Pilih `VISION_BACKEND = "gemini"` jika ingin memakai vision cloud.
2. Tambahkan Kaggle Secret bernama **GOOGLE_API_KEY** dan beri notebook akses. Gemini tetap dipakai untuk menulis narasi ketika vision memakai Qwen.
3. Ubah `I_HAVE_RIGHTS = True` hanya setelah memastikan hak penggunaan video.
4. Jalankan cell berurutan. Instalasi, download, TTS, dan rendering berlangsung di Kaggle.

Notebook ini disusun dari notebook YouTube Anda dan kode repository yang diperiksa pada 27 Oktober 2026 (commit acuan `f645cb9`). **Belum dieksekusi atau diuji audiovisual.** Link contoh baru dimasukkan sebagai input; keberhasilan download tidak dijamin.


## 1. Konfigurasi

- `VIDEO_INPUT`: URL/share text TikTok, RedNote/Xiaohongshu, atau path video lokal di Kaggle.
- `ENABLE_CANVAS = False`: hanya `final_id_source.mp4`, komposisi asli dipertahankan.
- `ENABLE_CANVAS = True`: tambah `final_id_canvas_9x16.mp4` ber-background blur. Output source tetap dibuat. Ini mengatur canvas **visual tambahan**; scheduler audio tetap aktif agar suara sinkron.
- `NARRATION_SOURCE = "visual"`: cerita dari kejadian visual. Pilih `"audio"` untuk konteks dialog/subtitle/ASR; ini tetap narration pipeline, bukan terjemahan dialog kata demi kata.
- `START_FROM`: awalnya `"segment"`; setelah kegagalan dapat diubah ke `"analyze"`, `"narrate"`, `"dub"`, atau `"render"` jika artefak tahap sebelumnya tersedia.
- `PROJECT_NAME = ""`: otomatis berdasarkan identitas video. Gunakan nama baru untuk eksperimen terpisah.


In [ ]:
from pathlib import Path
import os
import sys
import json
import re
import subprocess
import shutil
import time

REPO_URL = "https://github.com/NaufalRizqullah/dubbingstory.git"
GIT_REF = "feat/rednote-only-dubbing"
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "dubbingstory-social"
VENV_DIR = WORK_ROOT / ".dubbingstory-social-env"
OUTPUT_ROOT = WORK_ROOT / "dubbingstory_social_outputs"

VIDEO_INPUT = "https://www.rednote.com/discovery/item/6ad206ee000000001400240a"
# Alternatif: "https://www.tiktok.com/@creator/video/..." atau "/kaggle/input/dataset/video.mp4"
I_HAVE_RIGHTS = False
PROJECT_NAME = ""
DOWNLOAD_QUALITY = "max"  # 1080 membatasi TINGGI; max menjaga sumber portrait 1080x1920.

NARRATION_SOURCE = "visual"  # visual | audio
STYLE = "viral_fb"          # viral_fb | jozzr_social | documentary | technical | calm_educational
VOICE_ID = "id-ID-GadisNeural"
NARRATION_FIT = "rewrite"   # rewrite | error; video tidak dipercepat/dipotong.
FIT_RETRIES = 2
SPEECH_COVERAGE = 0.85
ORIGINAL_VOLUME = 0.15
NARRATION_VOLUME = 1.0

SUBTITLE_STYLE = "karaoke"   # karaoke | word | classic
BURN_SUBTITLES = True
WORD_ALIGNMENT = "edge"     # edge | auto | whisperx; selain edge memasang extra alignment.
CAPTION_ANIMATION = "pop"   # pop | fade | scale | none
FONT_NAME = "DejaVu Sans"
FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
FONT_SIZE = 64
FONT_COLOR = "#FFFFFF"
ACTIVE_COLOR = "#FFE600"
OUTLINE_COLOR = "#000000"
OUTLINE_WIDTH = 3
CAPTION_POSITION = "bottom"
SAFE_MARGIN_X = 0.08
SAFE_MARGIN_Y = 0.18
MAX_WORDS = 5

ENABLE_CANVAS = False
FOREGROUND_ZOOM = 25.0
BACKGROUND_ZOOM = 100.0
BACKGROUND_BLUR = 50.0
BACKGROUND_GRAYSCALE = 1.0

MIN_SCENE_DURATION = 2.0
MAX_SCENE_DURATION = 12.0
SCENE_THRESHOLD = 3.0
MAX_KEYFRAMES = 5
START_FROM = "segment"

VISION_BACKEND = "qwen"     # qwen | gemini
MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"
GEMINI_MODEL = "gemini-2.5-flash"
VLLM_VERSION = "0.27.1"     # Dipertahankan dari notebook asal; bukan klaim sudah diuji di Kaggle Anda.
VLLM_PARALLEL_MODE = "auto" # auto | data_parallel | tensor_parallel | single
GPU_MEMORY_UTILIZATION = 0.85
MAX_MODEL_LEN = 10240
VISION_MAX_TOKENS = 640
VISION_CONCURRENCY_OVERRIDE = None
IMAGE_MODE = "file"        # file | data; file diizinkan hanya di OUTPUT_ROOT.
PORT = 8000
SERVER_TIMEOUT_SECONDS = 900
WHISPER_MODEL = "base"     # Hanya dipakai jika NARRATION_SOURCE = audio.


## 2. Checkout dan runtime terpisah

Cell ini membuat folder repository khusus; **tidak menghapus isi working directory**. Checkout yang sudah ada dipakai kembali dan tidak di-pull/reset otomatis. Jika ingin branch/revision lain, gunakan folder `REPO_DIR` baru.

Dependency dipasang di virtual environment khusus agar kernel Kaggle tidak perlu mengimpor ulang PyTorch yang berubah. PyTorch mengikuti dependency vLLM, tanpa memaksa pasangan Torch/CUDA dari notebook lama. Gunakan sesi Kaggle baru jika driver/GPU tidak kompatibel.

Tidak ada cookies, Deno, client override YouTube, atau unduhan subtitle YouTube di workflow ini.


In [ ]:
if not Path("/kaggle").exists():
    raise RuntimeError("Notebook ini khusus Kaggle. Upload notebook ke Kaggle sebelum menjalankan cell.")
if VISION_BACKEND not in {"qwen", "gemini"} or NARRATION_SOURCE not in {"visual", "audio"}:
    raise ValueError("Periksa VISION_BACKEND dan NARRATION_SOURCE.")
if WORD_ALIGNMENT not in {"edge", "auto", "whisperx"}:
    raise ValueError("WORD_ALIGNMENT harus edge, auto, atau whisperx.")
if SUBTITLE_STYLE not in {"classic", "word", "karaoke"} or NARRATION_FIT not in {"rewrite", "error"}:
    raise ValueError("Periksa SUBTITLE_STYLE dan NARRATION_FIT.")
if START_FROM not in {"segment", "analyze", "narrate", "dub", "render"}:
    raise ValueError("START_FROM tidak dikenal.")
if not isinstance(ENABLE_CANVAS, bool):
    raise ValueError("ENABLE_CANVAS harus True atau False.")
if not I_HAVE_RIGHTS:
    raise ValueError("Konfirmasi hak penggunaan video melalui I_HAVE_RIGHTS sebelum menjalankan pipeline.")

WORK_ROOT.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("REPO_DIR sudah berisi folder non-Git. Pilih folder checkout baru.")
else:
    print("Memakai checkout existing; tidak melakukan pull/reset.")

REPO_REVISION = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Commit:", REPO_REVISION)

required_markers = {
    "dubbingstory/ingest/sources.py": ["normalize_source_url", '"tiktok"', '"xiaohongshu"'],
    "dubbingstory/cli.py": ["--narration-source", "--narration-fit", "--subtitle-style", "--vertical-layout"],
    "dubbingstory/tts/edge_tts_engine.py": ["WordBoundary"],
    "dubbingstory/tts/voice_manager.py": ["strict_timing_validated"],
    "dubbingstory/render/video_render.py": ["_canvas", "source_sha256"],
    "dubbingstory/render/audio_mix.py": ["channel_layouts=stereo", "sample_fmts=dbl"],
}
for relative, markers in required_markers.items():
    source_path = REPO_DIR / relative
    content = source_path.read_text(encoding="utf-8") if source_path.is_file() else ""
    if any(marker not in content for marker in markers):
        raise RuntimeError(f"Checkout belum memuat fitur/perbaikan yang dibutuhkan: {relative}. Gunakan branch terbaru di folder baru.")

if not shutil.which("ffmpeg") or not shutil.which("ffprobe") or not Path(FONT_PATH).is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "fonts-dejavu-core"], check=True)
if not Path(FONT_PATH).is_file():
    raise RuntimeError("FONT_PATH tidak ditemukan. Pilih font yang terpasang.")

RUNTIME_PYTHON = VENV_DIR / "bin" / "python"
if not RUNTIME_PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
subprocess.run([str(RUNTIME_PYTHON), "-m", "pip", "install", "--quiet", "--upgrade", "pip"], check=True)
extras = ["openai-vision"]
if WORD_ALIGNMENT != "edge" and SUBTITLE_STYLE != "classic":
    extras.append("alignment")
packages = ["-e", f"{REPO_DIR}[{','.join(extras)}]", "jsonschema>=4.21", "yt-dlp[default]"]
if VISION_BACKEND == "qwen" and START_FROM in {"segment", "analyze"}:
    packages.append(f"vllm=={VLLM_VERSION}")
if NARRATION_SOURCE == "audio":
    packages.append("faster-whisper")
subprocess.run([str(RUNTIME_PYTHON), "-m", "pip", "install", "--quiet", "--upgrade", *packages], check=True)
subprocess.run([str(RUNTIME_PYTHON), "-m", "pip", "check"], check=True)
subprocess.run([str(RUNTIME_PYTHON), "-c",
    "from importlib.metadata import version; print({p: version(p) for p in ('dubbingstory', 'yt-dlp', 'edge-tts')})"], check=True)

filters = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True, check=True)
if BURN_SUBTITLES and "subtitles" not in filters.stdout + filters.stderr:
    raise RuntimeError("FFmpeg ini tidak menyediakan filter subtitles/libass.")
print("Runtime terpasang. Belum memuat model vision atau menjalankan pipeline.")


## 3. Identitas sumber dan konfigurasi runtime

URL hanya diterima untuk TikTok atau RedNote/Xiaohongshu. File lokal tetap tersedia jika platform membatasi unduhan. Share text boleh berisi tepat satu URL. Normalisasi memakai adapter repository; query yang dibutuhkan extractor tidak dibuang.

Konfigurasi disimpan tanpa API key. Semua output berada di `/kaggle/working/dubbingstory_social_outputs/<project>/`.


In [ ]:
def render_options(enable_canvas):
    return {
        "ratios": ["source", "9:16"] if enable_canvas else ["source"],
        "vertical_layout": "layered" if enable_canvas else "pad",
    }

def expected_output_names(enable_canvas):
    names = ["final_id_source.mp4"]
    if enable_canvas:
        names.append("final_id_canvas_9x16.mp4")
    return names

def ingestion_arguments(video_input, is_url, project, quality, rights):
    if not rights:
        raise ValueError("Konfirmasi hak penggunaan video terlebih dahulu.")
    arguments = ["ingest", "--project", project, "--quality", quality]
    if is_url:
        arguments += ["--url", video_input, "--i-have-rights"]
    else:
        arguments += ["--input", video_input]
    return arguments

def pipeline_steps(project, start_from):
    # Ini jalur full existing: tidak memanggil summary/cut/concat.
    stages = [
        ("segment", ["segment", "--project", project]),
        ("analyze", ["analyze", "--project", project]),
        ("narrate", ["narrate", "--project", project, "--lang", "id"]),
        ("dub", ["dub", "--project", project, "--engine", "edge", "--voice-id", VOICE_ID]),
        ("render", ["render", "--project", project, "--mode", "full"]),
    ]
    names = [name for name, _ in stages]
    return stages[names.index(start_from):]

IS_URL = bool(re.search(r"https?://", VIDEO_INPUT, re.I))
if IS_URL:
    # Input melalui stdin, bukan log. Output hanya provider dan identity tanpa query.
    identity_code = """
import json, sys
from dubbingstory.ingest.sources import normalize_source_url, SourceIngestError
try:
    source = normalize_source_url(sys.stdin.read())
    if source.provider not in {"tiktok", "xiaohongshu"}:
        raise ValueError("Notebook ini khusus TikTok dan RedNote/Xiaohongshu.")
    print(json.dumps({"provider": source.provider, "project": source.identity.replace(":", "_")}))
except (SourceIngestError, ValueError) as exc:
    print(str(exc), file=sys.stderr)
    sys.exit(1)
"""
    identity_result = subprocess.run([str(RUNTIME_PYTHON), "-c", identity_code],
        input=VIDEO_INPUT, capture_output=True, text=True, cwd=REPO_DIR)
    if identity_result.returncode:
        raise RuntimeError(identity_result.stderr.strip() or "URL sumber tidak valid.")
    SOURCE_IDENTITY = json.loads(identity_result.stdout)
else:
    local_path = Path(VIDEO_INPUT).expanduser().resolve()
    if not local_path.is_file():
        raise FileNotFoundError("File lokal tidak ditemukan; isi path /kaggle/input/... yang benar.")
    import hashlib
    digest = hashlib.sha256()
    with local_path.open("rb") as media:
        for block in iter(lambda: media.read(1024 * 1024), b""):
            digest.update(block)
    VIDEO_INPUT = str(local_path)
    SOURCE_IDENTITY = {"provider": "local", "project": "local_" + digest.hexdigest()[:20]}

ACTIVE_PROJECT = PROJECT_NAME.strip() or SOURCE_IDENTITY["project"]
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,100}", ACTIVE_PROJECT):
    raise ValueError("PROJECT_NAME hanya boleh huruf, angka, underscore, dan tanda minus (maksimal 101 karakter).")
PROJECT_DIR = OUTPUT_ROOT / ACTIVE_PROJECT
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_DIR / "notebook_runtime.yaml"
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

CONFIG = {
    "requested_languages": ["id"],  # Filter cached scripts/audio in standalone dub/render too.
    "project": {"output_dir": str(OUTPUT_ROOT)},
    "ingest": {"download_height": DOWNLOAD_QUALITY, "read_subtitles": NARRATION_SOURCE == "audio",
               "use_asr_fallback": NARRATION_SOURCE == "audio"},
    "whisper": {"model": WHISPER_MODEL, "device": "cpu", "compute_type": "int8"},
    "segment": {"min_scene_duration": MIN_SCENE_DURATION, "max_scene_duration": MAX_SCENE_DURATION,
                "adaptive_threshold": SCENE_THRESHOLD},
    "keyframes": {"max_per_scene": MAX_KEYFRAMES},
    "vision": {"provider": "openai" if VISION_BACKEND == "qwen" else "gemini",
               "gemini_model": GEMINI_MODEL, "openai_model": MODEL_NAME, "openai_base_url": BASE_URL,
               "openai_image_mode": IMAGE_MODE, "openai_max_tokens": VISION_MAX_TOKENS,
               "openai_model_max_context": MAX_MODEL_LEN, "concurrency": 1,
               "openai_structured_outputs": True, "openai_allow_schema_fallback": False},
    "story": {"model": GEMINI_MODEL},
    "narration": {"languages": ["id"], "style": STYLE, "source": NARRATION_SOURCE,
                  "model": GEMINI_MODEL, "speech_coverage": SPEECH_COVERAGE},
    "tts": {"engine": "edge", "edge_voices": {"id": VOICE_ID}, "narration_fit": NARRATION_FIT,
            "fit_retries": FIT_RETRIES, "use_timeline_scheduler": True, "min_gap": 0.20},
    "render": {**render_options(ENABLE_CANVAS), "burn_subtitles": BURN_SUBTITLES,
               "audio_strategy": "dynamic_duck", "original_volume": ORIGINAL_VOLUME,
               "narration_volume": NARRATION_VOLUME, "foreground_zoom": FOREGROUND_ZOOM,
               "background_zoom": BACKGROUND_ZOOM, "background_blur": BACKGROUND_BLUR,
               "background_grayscale": BACKGROUND_GRAYSCALE},
    "subtitle": {"style": SUBTITLE_STYLE, "alignment": WORD_ALIGNMENT, "alignment_device": "cpu",
                 "font_name": FONT_NAME, "font_path": FONT_PATH, "font_size": FONT_SIZE,
                 "font_color": FONT_COLOR, "active_color": ACTIVE_COLOR, "outline_color": OUTLINE_COLOR,
                 "outline": OUTLINE_WIDTH, "position": CAPTION_POSITION, "animation": CAPTION_ANIMATION,
                 "safe_margin_x": SAFE_MARGIN_X, "safe_margin_y": SAFE_MARGIN_Y, "max_words": MAX_WORDS},
}

def save_config():
    # JSON merupakan subset YAML; loader repository membacanya sebagai YAML.
    CONFIG_PATH.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

save_config()
CHILD_ENV = os.environ.copy()
# Config notebook harus menjadi sumber pengaturan vision untuk subprocess.
for env_name in list(CHILD_ENV):
    if env_name.startswith("OPENAI_VISION_") or env_name in {"VISION_PROVIDER", "VISION_CONCURRENCY"}:
        CHILD_ENV.pop(env_name, None)
if VISION_BACKEND == "qwen":
    CHILD_ENV["OPENAI_VISION_API_KEY"] = "EMPTY"  # server lokal, bukan credential
CHILD_ENV["PYTHONUNBUFFERED"] = "1"

def safe_log(text):
    # Jangan menampilkan query share URL atau credential yang diberikan ke child.
    text = re.sub(r"(https?://[^\s?\"'<>]+)\?[^\s\"'<>]*", r"\1?[redacted]", str(text))
    for name in ("GOOGLE_API_KEY", "HF_TOKEN", "HF_API_KEY", "OPENAI_VISION_API_KEY"):
        value = CHILD_ENV.get(name, "")
        if value and value != "EMPTY":
            text = text.replace(value, "[redacted]")
    return text

def run_cli(arguments, label):
    command = [str(RUNTIME_PYTHON), "-u", str(REPO_DIR / "main.py"), "--config", str(CONFIG_PATH), *arguments]
    log_path = PROJECT_DIR / f"notebook_{label}.log"
    print(f"\n▶ {label} | project={ACTIVE_PROJECT}")
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, env=CHILD_ENV,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                clean = safe_log(line)
                print(clean, end="", flush=True)
                log.write(clean)
            returncode = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
    if returncode:
        raise RuntimeError(f"Tahap {label} gagal (exit {returncode}). Periksa {log_path.name}; jangan lanjut ke tahap berikutnya.")

print("Provider:", SOURCE_IDENTITY["provider"], "| Project:", ACTIVE_PROJECT)
print("Output:", ", ".join(expected_output_names(ENABLE_CANVAS)))
print("Canvas tambahan:", ENABLE_CANVAS, "| Konteks narasi:", NARRATION_SOURCE)


## 4. Ingest sumber — sebelum memuat Qwen

Adapter existing menangani normalisasi RedNote dan share link TikTok. Source disimpan sekali bersama identitas dan hash, lalu tahap berikutnya menggunakan project tersebut. Jangan mengganti URL remote menjadi `--input source.mp4` karena itu dapat mengubah provenance.

Jika unduhan dibatasi, privat, tidak tersedia, atau extractor gagal: **berhenti**, lalu gunakan video lokal yang Anda berhak gunakan melalui `VIDEO_INPUT` dan project baru. Notebook tidak melewati login, CAPTCHA, DRM, atau proteksi platform. Link contoh belum berhasil diverifikasi lewat akses web saat penyusunan; hasil ingest di Kaggle menjadi pemeriksaan aktual.


In [ ]:
run_cli(ingestion_arguments(VIDEO_INPUT, IS_URL, ACTIVE_PROJECT, DOWNLOAD_QUALITY, I_HAVE_RIGHTS), "ingest")
manifest_path = PROJECT_DIR / "ingest_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
metadata = json.loads((PROJECT_DIR / "video_metadata.json").read_text(encoding="utf-8"))
if manifest.get("status") != "ingested" or not (PROJECT_DIR / "source.mp4").is_file():
    raise RuntimeError("Ingest belum lengkap.")
print("Sumber siap:", metadata.get("provider"), "| Resolusi tampilan:",
      metadata.get("display_width"), "x", metadata.get("display_height"),
      "| Durasi video:", metadata.get("video_duration"))
if NARRATION_SOURCE == "audio" and not manifest.get("subtitle_data"):
    raise RuntimeError("Mode audio tidak mendapatkan transcript/subtitle. Periksa ASR, atau pilih mode visual secara eksplisit.")


## 5. Gemini Secret

Key diambil dari Kaggle Secrets dan diteruskan melalui environment subprocess. Tidak ditulis ke `.env`, YAML, output notebook, atau command line. Tidak diperlukan cookie platform atau key TTS untuk Edge.


In [ ]:
if START_FROM == "render":
    print("Render ulang tidak memerlukan Gemini Secret.")
else:
    from kaggle_secrets import UserSecretsClient
    try:
        gemini_key = UserSecretsClient().get_secret("GOOGLE_API_KEY").strip()
        if not gemini_key:
            raise ValueError("empty")
    except Exception:
        raise RuntimeError("Aktifkan Kaggle Secret GOOGLE_API_KEY untuk notebook ini.") from None
    CHILD_ENV["GOOGLE_API_KEY"] = gemini_key
    del gemini_key
    print("Gemini Secret tersedia; nilainya tidak ditampilkan.")


## 6. Jalankan dubbing full

Scene detection/keyframe hanya untuk analisis. Narasi dibatasi oleh timeline video; beat yang terlalu panjang diringkas dan TTS diukur ulang. Jika tetap tidak muat, proses gagal tanpa memotong video.

Qwen berjalan hanya selama analisis, kemudian dihentikan untuk membebaskan GPU. Rerun `narrate`, `dub`, atau `render` tidak memuat Qwen. Gunakan `START_FROM` sesuai artefak yang tersedia; mengubah teks/sumber/audio memerlukan regenerasi tahap terkait.

Boundary Edge harus mencakup semua kata. Angka/singkatan yang tidak sesuai ucapan perlu ditulis dalam bentuk yang dibacakan. Notebook tidak membagi durasi merata antar kata.


In [ ]:
import signal
import socket
import urllib.request

def stop_server(process):
    if process is None:
        return
    # vLLM dapat membuat worker GPU; hentikan seluruh group milik cell ini.
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait()

def qwen_command():
    gpu_probe = subprocess.run([str(RUNTIME_PYTHON), "-c",
        "import torch,json; print(json.dumps([{'name':torch.cuda.get_device_name(i),"
        "'gb':round(torch.cuda.get_device_properties(i).total_memory/1024**3,1),"
        "'cc':list(torch.cuda.get_device_capability(i))} for i in range(torch.cuda.device_count())]))"],
        capture_output=True, text=True, env=CHILD_ENV, check=False)
    if gpu_probe.returncode:
        raise RuntimeError("Runtime PyTorch tidak dapat membaca GPU. Periksa kompatibilitas wheel vLLM/driver, atau pilih VISION_BACKEND=gemini.")
    gpus = json.loads(gpu_probe.stdout)
    if not gpus:
        raise RuntimeError("GPU tidak ditemukan. Aktifkan GPU Kaggle atau pilih vision Gemini.")
    if min(gpu["cc"][0] for gpu in gpus) < 7:
        raise RuntimeError("GPU ini terlalu lama untuk preset vLLM. Pilih GPU yang didukung atau vision Gemini.")
    mode = VLLM_PARALLEL_MODE
    params = re.search(r"-(\d+(?:\.\d+)?)B(?:-|$)", MODEL_NAME, re.I)
    if mode == "auto":
        mode = "data_parallel" if len(gpus) > 1 and params and float(params.group(1)) <= 4.5 else (
            "tensor_parallel" if len(gpus) > 1 else "single")
    if mode not in {"single", "data_parallel", "tensor_parallel"}:
        raise ValueError("VLLM_PARALLEL_MODE tidak dikenal.")
    dp = len(gpus) if mode == "data_parallel" else 1
    tp = len(gpus) if mode == "tensor_parallel" else 1
    concurrency = dp if mode == "data_parallel" else min(2, tp)
    if VISION_CONCURRENCY_OVERRIDE is not None:
        concurrency = max(1, int(VISION_CONCURRENCY_OVERRIDE))
    CONFIG["vision"]["concurrency"] = concurrency
    save_config()
    command = [str(RUNTIME_PYTHON), "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME, "--host", "127.0.0.1", "--port", str(PORT),
        "--max-model-len", str(MAX_MODEL_LEN), "--tensor-parallel-size", str(tp),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION), "--dtype", "half",
        "--generation-config", "vllm", "--enforce-eager"]
    if dp > 1:
        command += ["--data-parallel-size", str(dp)]
    if IMAGE_MODE == "file":
        command += ["--allowed-local-media-path", str(OUTPUT_ROOT)]
    print("GPU:", gpus, "| Parallel mode:", mode, "| Vision concurrency:", concurrency)
    return command

def analyze_with_qwen(arguments):
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", PORT)) == 0:
            raise RuntimeError("PORT sudah dipakai. Hentikan server lama atau pilih PORT lain; server existing tidak diambil alih.")
    command = qwen_command()
    server_log_path = PROJECT_DIR / "vllm_server.log"
    server_env = CHILD_ENV.copy()
    # Gemini hanya diperlukan subprocess pipeline, bukan server vision lokal.
    server_env.pop("GOOGLE_API_KEY", None)
    server_env.pop("OPENAI_VISION_API_KEY", None)
    process = None
    with server_log_path.open("w", encoding="utf-8") as server_log:
        try:
            process = subprocess.Popen(command, cwd=REPO_DIR, env=server_env,
                stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
            deadline = time.monotonic() + SERVER_TIMEOUT_SECONDS
            ready = False
            while time.monotonic() < deadline:
                if process.poll() is not None:
                    raise RuntimeError("vLLM berhenti sebelum siap. Periksa vllm_server.log.")
                try:
                    with urllib.request.urlopen(f"{BASE_URL}/models", timeout=5) as response:
                        models = json.load(response).get("data", [])
                    if any(model.get("id") == MODEL_NAME for model in models):
                        ready = True
                        break
                except (OSError, ValueError):
                    pass
                time.sleep(5)
            if not ready:
                raise RuntimeError("vLLM belum siap sebelum timeout. Periksa log; jangan melanjutkan analisis.")
            run_cli(arguments, "analyze")
        finally:
            stop_server(process)

steps = pipeline_steps(ACTIVE_PROJECT, START_FROM)
required_artifact = {
    "segment": "source.mp4", "analyze": "segment_manifest.json",
    "narrate": "storyboard.json", "dub": "scripts/script_id.json",
    "render": "audio/audio_id.manifest.json",
}[START_FROM]
if not (PROJECT_DIR / required_artifact).is_file():
    raise RuntimeError(f"START_FROM={START_FROM} memerlukan {required_artifact}. Jalankan tahap sebelumnya.")
for stage, arguments in steps:
    if stage == "analyze" and VISION_BACKEND == "qwen":
        analyze_with_qwen(arguments)
    else:
        run_cli(arguments, stage)
print("\nPipeline full selesai. Periksa manifest dan preview pada cell berikutnya.")


## 7. Hasil dan file download

Preview hanya menampilkan output yang diminta konfigurasi saat ini, sehingga canvas lama tidak tampil ketika toggle dimatikan. ZIP berisi MP4 final, voice-over WAV, subtitle, naskah final, word timing, dan manifest; video sumber serta model tidak ikut dibundel.

Pemeriksaan visual tetap diperlukan: frame awal/akhir dan komposisi sama, narasi tidak terpotong, kata mengikuti suara, safe area tidak menutupi objek penting, serta ducking pulih saat narator berhenti.


In [ ]:
from IPython.display import display, Video, FileLink
import zipfile

render_manifest = json.loads((PROJECT_DIR / "render_manifest.json").read_text(encoding="utf-8"))
if render_manifest.get("status") != "complete":
    raise RuntimeError("Render belum lengkap. Periksa render_manifest.json dan log render.")
expected = [PROJECT_DIR / name for name in expected_output_names(ENABLE_CANVAS)]
reported = {Path(path).resolve() for path in render_manifest.get("rendered_videos", [])}
if any(not path.is_file() or path.resolve() not in reported for path in expected):
    raise RuntimeError("Output yang diminta tidak cocok dengan render_manifest. Jalankan render ulang.")

# FileLink relatif ke /kaggle/working agar bisa diunduh melalui notebook Kaggle.
os.chdir(WORK_ROOT)
for path in expected:
    relative = path.relative_to(WORK_ROOT)
    display(FileLink(str(relative)))
    display(Video(filename=str(path), embed=True, width=360))

bundle_path = WORK_ROOT / f"{ACTIVE_PROJECT}_dubbing_results.zip"
artifacts = expected + [
    PROJECT_DIR / "scripts" / f"script_id{suffix}"
    for suffix in (".txt", ".json", ".srt", ".words.json", ".ass")
] + [
    PROJECT_DIR / "ingest_manifest.json", PROJECT_DIR / "video_metadata.json",
    PROJECT_DIR / "audio" / "audio_id.wav",
    PROJECT_DIR / "audio" / "audio_id.manifest.json", PROJECT_DIR / "render_manifest.json",
    CONFIG_PATH,
]
artifacts += sorted((PROJECT_DIR / "scripts").glob("script_id.*.ass"))
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in dict.fromkeys(artifacts):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(PROJECT_DIR)))
print("Bundle hasil:")
display(FileLink(bundle_path.name))
print("Durasi sumber dan sinkronisasi caption tetap perlu diperiksa secara audiovisual.")


## 8. Opsional — render ulang tanpa vision/TTS

Ubah toggle/style di cell berikutnya, lalu jalankan. Memilih canvas mengubah komposisi **output tambahan** saja. Karaoke/word membutuhkan file timing yang sudah dibuat; jika awalnya memakai classic, ulangi `dub` dengan animated style lebih dulu.

Untuk perubahan naskah atau voice, ulangi tahap `dub` dan `render`, bukan hanya cell render ulang. File dengan nama output sama dapat diganti.


In [ ]:
RUN_RENDER_AGAIN = False
RERENDER_CANVAS = ENABLE_CANVAS
RERENDER_SUBTITLE_STYLE = SUBTITLE_STYLE

if RUN_RENDER_AGAIN:
    if RERENDER_SUBTITLE_STYLE not in {"classic", "word", "karaoke"}:
        raise ValueError("Style tidak dikenal.")
    ENABLE_CANVAS = bool(RERENDER_CANVAS)
    CONFIG["render"].update(render_options(ENABLE_CANVAS))
    CONFIG["subtitle"]["style"] = RERENDER_SUBTITLE_STYLE
    save_config()
    run_cli(["render", "--project", ACTIVE_PROJECT, "--mode", "full"], "render")
    print("Jalankan kembali cell hasil untuk memperbarui preview dan ZIP.")
else:
    print("Render ulang dilewati.")


## Pengujian dan catatan runtime

Notebook baru tidak menjalankan test suite otomatis. Untuk memeriksa regression tests **di Kaggle**, gunakan checkout yang sudah memuat perubahan notebook dan tests baru, lalu jalankan melalui terminal notebook setelah setup:

```bash
/kaggle/working/.dubbingstory-social-env/bin/python -m pip install -e '/kaggle/working/dubbingstory-social[test]'
cd /kaggle/working/dubbingstory-social
/kaggle/working/.dubbingstory-social-env/bin/python -m pytest -q tests
```

- Download gagal: lihat kategori ingest; gunakan file lokal yang Anda berhak gunakan bila platform menolak. Tidak ada fallback yang melewati proteksi.
- Qwen gagal/OOM: periksa driver dan vLLM log; turunkan `MAX_MODEL_LEN`, pakai model lebih kecil yang didukung, atau gunakan `VISION_BACKEND="gemini"`. Instalasi membutuhkan ruang untuk dependency serta model.
- Overflow narasi: `rewrite` mencoba ringkas per beat dengan batas retry. `error` berhenti segera. Video tidak diperpanjang, dipotong, atau dipercepat.
- Alignment Edge gagal: koreksi angka/singkatan menjadi bentuk ucapan; atau pilih `WORD_ALIGNMENT="whisperx"`/`"auto"` lalu ulangi setup dan dub. Extra ini membawa model/dependency tambahan; kompatibilitas dengan vLLM diperiksa resolver saat instalasi, bukan diasumsikan.
- Burn subtitle perlu libass dan re-encoding. Tanpa burn/filter visual, codec yang kompatibel dapat di-stream-copy. Encoding HDR ditolak sampai kebijakan color management tersedia.
- Rerun ingest menjaga cache dan menolak sumber berbeda dalam project sama. Mengubah `VIDEO_INPUT` mengharuskan mengulang cell identitas/config dan ingest.
- Jangan membagikan log server atau metadata mentah tanpa meninjau isinya; notebook tidak menyalin output/kredensial dari notebook lama.

Referensi implementasi: `dubbingstory/ingest/sources.py`, `dubbingstory/cli.py`, `dubbingstory/tts/edge_tts_engine.py`, `dubbingstory/tts/voice_manager.py`, `dubbingstory/render/video_render.py`, dan `docs/native-storytelling.md`.

Runtime terpisah mengikuti [panduan instalasi GPU vLLM](https://docs.vllm.ai/en/latest/getting_started/installation/gpu/). Pin vLLM berasal dari notebook sumber; dukungan wheel/driver/GPU tetap harus diverifikasi saat dijalankan.
